# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [2]:
# Load the raw NTSB data. The file is not UTF-8 encoded, so latin-1 is used.
df = pd.read_csv("data/AviationData.csv", encoding="latin-1", low_memory=False)
print("Shape:", df.shape)
df.head()

Shape: (88889, 31)


,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Purpose.of.flight,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date
0,20001218X45444,Accident,SEA87LA080,1948-10-24,"MOOSE CREEK, ID",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,UNK,Cruise,Probable Cause,NaN
1,20001218X45447,Accident,LAX94LA336,1962-07-19,"BRIDGEPORT, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,4.0,0.0,0.0,0.0,UNK,Unknown,Probable Cause,19-09-1996
2,20061025X01555,Accident,NYC07LA005,1974-08-30,"Saltville, VA",United States,36.922223,-81.878056,NaN,NaN,...,Personal,NaN,3.0,NaN,NaN,NaN,IMC,Cruise,Probable Cause,26-02-2007
3,20001218X45448,Accident,LAX96LA321,1977-06-19,"EUREKA, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,IMC,Cruise,Probable Cause,12-09-2000
4,20041105X01764,Accident,CHI79FA064,1979-08-02,"Canton, OH",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,1.0,2.0,NaN,0.0,VMC,Approach,Probable Cause,16-04-1980


In [3]:
# Datatypes and non-null counts
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                88889 non-null  str    
 1   Investigation.Type      88889 non-null  str    
 2   Accident.Number         88889 non-null  str    
 3   Event.Date              88889 non-null  str    
 4   Location                88837 non-null  str    
 5   Country                 88663 non-null  str    
 6   Latitude                34382 non-null  str    
 7   Longitude               34373 non-null  str    
 8   Airport.Code            50132 non-null  str    
 9   Airport.Name            52704 non-null  str    
 10  Injury.Severity         87889 non-null  str    
 11  Aircraft.damage         85695 non-null  str    
 12  Aircraft.Category       32287 non-null  str    
 13  Registration.Number     87507 non-null  str    
 14  Make                    88826 non-null  str    
 

In [4]:
# Missing values per column (count and %)
na = pd.DataFrame({"n_missing": df.isna().sum(), "pct_missing": (df.isna().mean() * 100).round(1)})
na.sort_values("n_missing", ascending=False)

,n_missing,pct_missing
Schedule,76307,85.8
Air.carrier,72241,81.3
FAR.Description,56866,64.0
Aircraft.Category,56602,63.7
Longitude,54516,61.3
Latitude,54507,61.3
Airport.Code,38757,43.6
Airport.Name,36185,40.7
Broad.phase.of.flight,27165,30.6
Publication.Date,13771,15.5


In [5]:
# Summary statistics: numeric and categorical
display(df.describe().T)
display(df.describe(include="object").T)

,count,mean,std,min,25%,50%,75%,max
Number.of.Engines,82805.0,1.146585,0.446510,0.0,1.0,1.0,1.0,8.0
Total.Fatal.Injuries,77488.0,0.647855,5.485960,0.0,0.0,0.0,0.0,349.0
Total.Serious.Injuries,76379.0,0.279881,1.544084,0.0,0.0,0.0,0.0,161.0
Total.Minor.Injuries,76956.0,0.357061,2.235625,0.0,0.0,0.0,0.0,380.0
Total.Uninjured,82977.0,5.325440,27.913634,0.0,0.0,1.0,2.0,699.0


C:\Users\Administrator\AppData\Local\Temp\ipykernel_5924\740803171.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  display(df.describe(include="object").T)


,count,unique,top,freq
Event.Id,88889,87951,20001214X45071,3
Investigation.Type,88889,2,Accident,85015
Accident.Number,88889,88863,ERA22LA103,2
Event.Date,88889,14782,1982-05-16,25
Location,88837,27758,"ANCHORAGE, AK",434
Country,88663,219,United States,82248
Latitude,34382,25589,332739N,19
Longitude,34373,27154,0112457W,24
Airport.Code,50132,10374,NONE,1488
Airport.Name,52704,24870,Private,240


**Initial observations**
- 88,889 rows and 31 columns; `Event.Date` is stored as text and must be converted to a datetime.
- `Aircraft.Category` is missing for ~64% of rows. This matters a lot: the client wants *airplanes*, and the field is almost entirely blank for events before 2008. We will have to impute it.
- Injury columns are missing for 6-14% of rows, and several categorical columns hold placeholders (`Unknown`, `UNK`, `Unk`) that are really missing values.
- Latitude/longitude, airport, air carrier, FAR description and schedule are mostly empty and are not needed for this analysis.

## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

### Filtering steps and rationale
1. **Date**: keep events from **1983-01-01** onward (a 40-year maximum service life means older designs are retired).
2. **Investigation type**: keep **Accident** only. The client asks about outcomes "in the event of an accident"; NTSB *incidents* are by definition lower-severity events and would bias the destruction/injury rates downward.
3. **Professional builds**: keep `Amateur.Built == "No"`; drop the 101 rows where it is missing since we cannot confirm them as professional builds.
4. **Airplanes only**: the brief is about aircraft/passenger airlines, not helicopters, balloons, gliders etc. Since `Aircraft.Category` is blank for most pre-2008 events we impute it (see below) before filtering.

In [6]:
df["Event.Date"] = pd.to_datetime(df["Event.Date"])
print("Date range:", df["Event.Date"].min().date(), "->", df["Event.Date"].max().date())

n0 = len(df)
df = df[df["Event.Date"] >= "1983-01-01"].copy()
print(f"After date filter (>=1983): {len(df):,} (dropped {n0 - len(df):,})")

n0 = len(df)
df = df[df["Investigation.Type"] == "Accident"].copy()
print(f"After keeping Accidents only: {len(df):,} (dropped {n0 - len(df):,})")

n0 = len(df)
df = df[df["Amateur.Built"] == "No"].copy()
print(f"After keeping professional builds: {len(df):,} (dropped {n0 - len(df):,})")

Date range: 1948-10-24 -> 2022-12-29
After date filter (>=1983): 85,289 (dropped 3,600)
After keeping Accidents only: 81,534 (dropped 3,755)
After keeping professional builds: 73,286 (dropped 8,248)


**Light normalisation of `Make` / `Model` (needed for the imputation)**: upper-case and strip whitespace, and drop rows with a missing Make or Model, since an aircraft type cannot be identified without them. The deeper Make cleaning follows in its own section.

In [7]:
df["Make"] = df["Make"].str.upper().str.strip()
df["Model"] = df["Model"].str.upper().str.strip()
n0 = len(df)
df = df.dropna(subset=["Make", "Model"]).copy()
print(f"Dropped {n0 - len(df)} rows with missing Make/Model -> {len(df):,} rows")

Dropped 39 rows with missing Make/Model -> 73,247 rows


**Imputing `Aircraft.Category`**

Placeholders (`UNK`, `Unknown`) and abbreviations (`WSFT`, `ULTR`) are first standardised. Then missing categories are imputed in three passes, from most to least specific:
1. **Make + Model lookup** - use the most common known category of the same Make/Model in the dataset (e.g. a "CESSNA 172M" with no category inherits "Airplane" from other 172Ms).
2. **Make lookup** - if the Make has a known category in >=95% of its labelled events (e.g. Piper), use it.
3. **Engine type rule** - Turbo Fan / Turbo Jet / Turbo Prop engines => Airplane, Turbo Shaft => Helicopter. (This rescues many jets, e.g. McDonnell Douglas, whose Make is split between helicopters and airplanes.)

Rows still unresolved are dropped, because we cannot tell whether they are airplanes.

In [8]:
cat = df["Aircraft.Category"].replace({"UNK": np.nan, "Unknown": np.nan, "WSFT": "Weight-Shift", "ULTR": "Ultralight"})
print("Missing category before imputation:", cat.isna().sum())
known = df.assign(cat=cat).dropna(subset=["cat"])

# Pass 1: Make + Model mode
mm = known.groupby(["Make", "Model"])["cat"].agg(lambda s: s.value_counts().index[0])
key = pd.MultiIndex.from_frame(df[["Make", "Model"]])
cat = cat.fillna(pd.Series(mm.reindex(key).values, index=df.index))
print("After Make+Model pass:", cat.isna().sum())

# Pass 2: Make mode, only if the make is >=95% one category
mk_mode = known.groupby("Make")["cat"].agg(lambda s: s.value_counts().index[0])
mk_pur = known.groupby("Make")["cat"].agg(lambda s: s.value_counts(normalize=True).iloc[0])
cat = cat.fillna(df["Make"].map(mk_mode[mk_pur >= 0.95]))
print("After Make pass:", cat.isna().sum())

# Pass 3: engine type rule
eng_rule = df["Engine.Type"].map({"Turbo Fan": "Airplane", "Turbo Jet": "Airplane", "Turbo Prop": "Airplane", "Turbo Shaft": "Helicopter"})
cat = cat.fillna(eng_rule)
print("After engine-type pass:", cat.isna().sum())

df["Aircraft.Category"] = cat
print(df["Aircraft.Category"].value_counts(dropna=False))

Missing category before imputation: 49472
After Make+Model pass: 7453
After Make pass: 1192
After engine-type pass: 996
Aircraft.Category
Airplane             62925
Helicopter            7567
NaN                    996
Glider                 984
Balloon                471
Weight-Shift           149
Powered Parachute       86
Gyrocraft               40
Ultralight              19
Blimp                    5
Powered-Lift             4
Rocket                   1
Name: count, dtype: int64


In [9]:
n0 = len(df)
df = df[df["Aircraft.Category"] == "Airplane"].copy()
print(f"Kept airplanes only: {len(df):,} (dropped {n0 - len(df):,} non-airplane / unresolved rows)")

Kept airplanes only: 62,925 (dropped 10,322 non-airplane / unresolved rows)


### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

### Fatal / serious injury fraction

**Assumptions**
- The four injury columns (`Fatal`, `Serious`, `Minor`, `Uninjured`) partition everyone on board, so **total on board = Fatal + Serious + Minor + Uninjured**. Note this includes crew, so it is "persons on board" - the best available proxy for passengers.
- If `Total.Fatal.Injuries` is missing but `Injury.Severity` is of the form `Fatal(n)`, we recover n from that field.
- If at least one injury column is reported, the remaining blanks are treated as **0** (NTSB leaves a count blank when nobody fell in that category). If *all four* are blank we have no information and drop the row.
- Rows with 0 people on board cannot define a fraction and are dropped.

**Derived columns**: `Total.Aboard`, `Fatal.Serious` (fatal + serious count), `Fatal.Serious.Frac` (= `Fatal.Serious / Total.Aboard`, between 0 and 1).

In [10]:
inj_cols = ["Total.Fatal.Injuries", "Total.Serious.Injuries", "Total.Minor.Injuries", "Total.Uninjured"]

# recover fatal count from Injury.Severity "Fatal(n)" when the numeric column is missing
sev_fatal = pd.to_numeric(df["Injury.Severity"].str.extract(r"Fatal\((\d+)\)")[0], errors="coerce")
mask = df["Total.Fatal.Injuries"].isna() & sev_fatal.notna()
print("Fatal counts recovered from Injury.Severity:", mask.sum())
df.loc[mask, "Total.Fatal.Injuries"] = sev_fatal[mask]

# drop rows with no injury information at all
n0 = len(df)
df = df.dropna(subset=inj_cols, how="all").copy()
print(f"Dropped {n0 - len(df):,} rows with all four injury columns missing")

# remaining blanks -> 0
df[inj_cols] = df[inj_cols].fillna(0)

df["Total.Aboard"] = df[inj_cols].sum(axis=1)
n0 = len(df)
df = df[df["Total.Aboard"] > 0].copy()
print(f"Dropped {n0 - len(df):,} rows with zero persons on board")

df["Fatal.Serious"] = df["Total.Fatal.Injuries"] + df["Total.Serious.Injuries"]
df["Fatal.Serious.Frac"] = df["Fatal.Serious"] / df["Total.Aboard"]
print("Rows:", len(df))
df[["Total.Aboard", "Fatal.Serious", "Fatal.Serious.Frac"]].describe()

Fatal counts recovered from Injury.Severity: 0
Dropped 48 rows with all four injury columns missing
Dropped 285 rows with zero persons on board
Rows: 62592


,Total.Aboard,Fatal.Serious,Fatal.Serious.Frac
count,62592.000000,62592.000000,62592.000000
mean,4.880592,0.885864,0.279941
std,23.290367,6.291346,0.433123
min,1.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000
50%,2.000000,0.000000,0.000000
75%,2.000000,1.000000,1.000000
max,699.000000,349.000000,1.000000


In [11]:
# sanity check: the fraction is bounded and the distribution is heavily bimodal (most crashes: 0, some: 1)
assert df["Fatal.Serious.Frac"].between(0, 1).all()
print(df["Fatal.Serious.Frac"].round(1).value_counts().sort_index())

Fatal.Serious.Frac
0.0    43003
0.1      139
0.2      425
0.3      538
0.4       77
0.5     2032
0.6       58
0.7      360
0.8      219
0.9       38
1.0    15703
Name: count, dtype: int64


**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

### Aircraft damage

`Aircraft.damage` has values `Substantial`, `Destroyed`, `Minor`, `Unknown`, and blanks. `Unknown` is a placeholder for missing and is converted to NaN. We do **not** impute the rest: the damage outcome is the very quantity the client wants to measure, so guessing it would contaminate the result. Rows with missing damage stay in the dataset (they are still valid for the injury analysis) and are excluded only from destruction-rate calculations.

**Derived column**: `Destroyed` = 1 if damage is `Destroyed`, 0 for `Substantial`/`Minor`, NaN if unknown.

In [12]:
print(df["Aircraft.damage"].value_counts(dropna=False))
df["Aircraft.damage"] = df["Aircraft.damage"].replace("Unknown", np.nan)

df["Destroyed"] = np.where(df["Aircraft.damage"].isna(), np.nan, (df["Aircraft.damage"] == "Destroyed").astype(float))
print()
print("Destroyed value counts:")
print(df["Destroyed"].value_counts(dropna=False))
print("Overall destroyed rate:", round(df["Destroyed"].mean(), 3))

Aircraft.damage
Substantial    47833
Destroyed      13258
NaN              950
Minor            487
Unknown           64
Name: count, dtype: int64

Destroyed value counts:
Destroyed
0.0    48320
1.0    13258
NaN     1014
Name: count, dtype: int64
Overall destroyed rate: 0.215


### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

### Make cleaning tasks
Listing the most frequent makes shows many spelling variants of the same manufacturer, which would split their sample sizes and distort the "at least 50 events" filter.

Cleaning tasks:
1. Upper-case and strip whitespace (done earlier).
2. Remove punctuation (`.` `,`) and collapse repeated spaces.
3. Remove corporate suffixes/words (`INC`, `CORP`, `COMPANY`, `CO`, `LTD`, `INDUSTRIES`, `DESIGN`, etc.) so "AIR TRACTOR INC" = "AIR TRACTOR".
4. Merge known aliases for the same manufacturer, e.g. `BEECHCRAFT` -> `BEECH`, `ROBINSON HELICOPTER` -> `ROBINSON`, `GRUMMAN AMERICAN` -> `GRUMMAN`, `AIRBUS INDUSTRIE` -> `AIRBUS`, `DEHAVILLAND` -> `DE HAVILLAND`, `DOUGLAS` -> `MCDONNELL DOUGLAS` (McDonnell merged with Douglas in 1967; they are one lineage of airliners such as DC-9/MD-80).
5. Merge remaining prefix variants (e.g. `ROCKWELL INTERNATIONAL` -> `ROCKWELL`, `GATES LEARJET` -> `LEARJET`, `AVIAT AIRCRAFT` -> `AVIAT`, `RAYTHEON AIRCRAFT` -> `BEECH` as Raytheon was the Beechcraft successor).
6. Keep only makes with **at least 50 events**.

In [13]:
print("Unique makes before cleaning:", df["Make"].nunique())
print(df["Make"].value_counts().head(40))

Unique makes before cleaning: 1082
Make
CESSNA                            25325
PIPER                             13899
BEECH                              4901
MOONEY                             1247
BOEING                             1214
GRUMMAN                            1051
BELLANCA                            972
AIR TRACTOR                         666
AERONCA                             605
MAULE                               569
CHAMPION                            502
STINSON                             418
AERO COMMANDER                      403
LUSCOMBE                            390
TAYLORCRAFT                         366
NORTH AMERICAN                      364
DE HAVILLAND                        359
ROCKWELL                            334
AYRES                               226
AIR TRACTOR INC                     217
CIRRUS DESIGN CORP                  212
GRUMMAN AMERICAN                    210
MCDONNELL DOUGLAS                   209
DOUGLAS                             181


In [14]:
import re

def clean_make(m):
    m = re.sub(r"[.,]", "", m)
    m = re.sub(r"\b(INC|CORP|CORPORATION|COMPANY|CO|LTD|LLC|INDUSTRIE|INDUSTRIES|DESIGN)\b", "", m)
    m = re.sub(r"\(.*?\)", "", m)        # drop parentheticals
    m = re.sub(r"\s+", " ", m).strip()
    return m

df["Make"] = df["Make"].map(clean_make)

aliases = {
    "BEECHCRAFT": "BEECH", "ROBINSON HELICOPTER": "ROBINSON", "GRUMMAN AMERICAN": "GRUMMAN",
    "AMERICAN GRUMMAN": "GRUMMAN", "DEHAVILLAND": "DE HAVILLAND", "DOUGLAS": "MCDONNELL DOUGLAS",
    "BOEING OF CANADA": "BOEING", "PIPER AIRCRAFT": "PIPER", "CESSNA AIRCRAFT": "CESSNA",
    "ERCOUPE ENG & RESEARCH": "ERCOUPE", "ENG & RESEARCH": "ERCOUPE",
    "AEROSPATIALE/ALENIA": "ATR", "AEROSPATIALE": "AEROSPATIALE",
}
df["Make"] = df["Make"].replace(aliases)
# prefix-based merges for the remaining spelling variants
prefix_rules = {
    r"^ERCOUPE.*": "ERCOUPE", r"^CIRRUS.*": "CIRRUS", r"^AIR TRACTOR.*": "AIR TRACTOR",
    r"^ROCKWELL.*": "ROCKWELL", r"^GULFSTREAM.*": "GULFSTREAM", r"^AVIAT.*": "AVIAT",
    r"^GATES LEARJET.*": "LEARJET", r"^LEARJET.*": "LEARJET", r"^DIAMOND.*": "DIAMOND",
    r"^AMERICAN CHAMPION.*": "CHAMPION", r"^GRUMMAN AMERICAN.*": "GRUMMAN",
    r"^GRUMMAN ACFT.*": "GRUMMAN-SCHWEIZER", r"^RAYTHEON.*": "BEECH",   # Raytheon Aircraft = Beechcraft successor
    r"^BOEING STEARMAN.*": "BOEING STEARMAN",
}
for pat, rep in prefix_rules.items():
    df["Make"] = df["Make"].str.replace(pat, rep, regex=True)
print("Unique makes after cleaning:", df["Make"].nunique())
print(df["Make"].value_counts().head(40))

Unique makes after cleaning: 920
Make
CESSNA               25369
PIPER                13945
BEECH                 5024
GRUMMAN               1316
MOONEY                1247
BOEING                1216
BELLANCA               972
AIR TRACTOR            891
AERONCA                605
CHAMPION               603
MAULE                  569
DE HAVILLAND           458
CIRRUS                 436
STINSON                418
ROCKWELL               406
AERO COMMANDER         403
MCDONNELL DOUGLAS      391
LUSCOMBE               390
TAYLORCRAFT            368
NORTH AMERICAN         364
AYRES                  272
ERCOUPE                232
AVIAT                  230
GULFSTREAM             190
LEARJET                188
GRUMMAN-SCHWEIZER      188
DIAMOND                161
AIRBUS                 155
EMBRAER                141
WACO                   137
LAKE                   137
SWEARINGEN             127
SCHWEIZER              127
SOCATA                 126
MITSUBISHI             125
FAIRCHILD        

In [15]:
# Keep makes with at least 50 events
make_counts = df["Make"].value_counts()
keep_makes = make_counts[make_counts >= 50].index
n0 = len(df)
df = df[df["Make"].isin(keep_makes)].copy()
print(f"{len(keep_makes)} makes retained; rows {n0:,} -> {len(df):,}")
make_counts[make_counts >= 50]

50 makes retained; rows 62,592 -> 59,150


Make
CESSNA               25369
PIPER                13945
BEECH                 5024
GRUMMAN               1316
MOONEY                1247
BOEING                1216
BELLANCA               972
AIR TRACTOR            891
AERONCA                605
CHAMPION               603
MAULE                  569
DE HAVILLAND           458
CIRRUS                 436
STINSON                418
ROCKWELL               406
AERO COMMANDER         403
MCDONNELL DOUGLAS      391
LUSCOMBE               390
TAYLORCRAFT            368
NORTH AMERICAN         364
AYRES                  272
ERCOUPE                232
AVIAT                  230
GULFSTREAM             190
LEARJET                188
GRUMMAN-SCHWEIZER      188
DIAMOND                161
AIRBUS                 155
EMBRAER                141
WACO                   137
LAKE                   137
SWEARINGEN             127
SCHWEIZER              127
SOCATA                 126
MITSUBISHI             125
FAIRCHILD              123
RYAN                   

### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

### Model cleaning
- Missing models were already dropped.
- Model labels are written inconsistently (`PA-28-181`, `PA28-181`, `PA 28 181`), so spaces and hyphens are removed to merge these variants.
- A model label is **not** unique to one manufacturer (e.g. numeric names such as `100` or `200` can occur across makes), so we create `Plane.Type = Make + " " + Model` as the unique identifier.

In [16]:
df["Model"] = df["Model"].str.replace(r"[\s\-]", "", regex=True)
df = df[df["Model"] != ""].copy()
print("NaN models:", df["Model"].isna().sum())

# are model labels unique to a make?
mk_per_model = df.groupby("Model")["Make"].nunique()
print("Model labels shared by more than one Make:", (mk_per_model > 1).sum(), "of", len(mk_per_model))
print(df[df["Model"].isin(mk_per_model[mk_per_model > 1].index)].groupby("Model")["Make"].unique().head(10))

df["Plane.Type"] = df["Make"] + " " + df["Model"]
print("\nUnique plane types:", df["Plane.Type"].nunique())
df["Plane.Type"].value_counts().head(15)

NaN models: 0
Model labels shared by more than one Make: 166 of 2960
Model
100            [AERO COMMANDER, ROCKWELL, BEECH]
100180                [AERO COMMANDER, ROCKWELL]
112       [AERO COMMANDER, GULFSTREAM, ROCKWELL]
112A                  [ROCKWELL, AERO COMMANDER]
112TCA                    [ROCKWELL, GULFSTREAM]
120                            [CESSNA, EMBRAER]
140                            [CESSNA, EMBRAER]
164                 [GRUMMAN, GRUMMAN-SCHWEIZER]
164B                [GRUMMAN, GRUMMAN-SCHWEIZER]
170                            [CESSNA, EMBRAER]
Name: Make, dtype: object

Unique plane types: 3172


Plane.Type
CESSNA 152       2213
CESSNA 172       1628
CESSNA 172N      1096
PIPER PA28140     929
CESSNA 172M       784
CESSNA 150        742
CESSNA 172P       666
PIPER PA18        665
PIPER PA18150     620
CESSNA 182        608
CESSNA 180        594
PIPER PA28180     592
PIPER PA28161     556
CESSNA 150M       551
PIPER PA28181     545
Name: count, dtype: int64

### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

### Cleaning the other factor columns

| Column | Issue | Action |
|---|---|---|
| `Engine.Type` | `UNK`, `Unknown`, `LR`, `NONE` placeholders; `Geared Turbofan` is a Turbo Fan; tiny categories (Electric, Hybrid Rocket) | Placeholders -> NaN, Geared Turbofan -> Turbo Fan, rare types -> `Other` |
| `Weather.Condition` | `UNK`, `Unk`, `Unknown` placeholders | -> NaN (only `VMC`/`IMC` kept) |
| `Number.of.Engines` | `0` engines is impossible for a powered airplane; 6 and 8 engines are extremely rare | `0` -> NaN, then impute from the Make+Model mode when available; 3+ engines grouped as `3+` in analysis |
| `Purpose.of.flight` | duplicate spellings (`Air Race show`, `Air Race/show`, `ASHO`), many `Public Aircraft - X` sub-types, `Unknown` placeholders, tiny categories | Merge duplicates, merge Public Aircraft sub-types, `Unknown` -> NaN, categories with < 50 rows -> `Other` |
| `Broad.phase.of.flight` | `Unknown` / `Other` placeholders | `Unknown` -> NaN |

We do not drop NaNs here: each analysis will drop only what it needs.

In [17]:
for c in ["Engine.Type", "Weather.Condition", "Number.of.Engines", "Purpose.of.flight", "Broad.phase.of.flight"]:
    print(f"--- {c}")
    print(df[c].value_counts(dropna=False))
    print()

--- Engine.Type
Engine.Type
Reciprocating    51603
NaN               3063
Turbo Prop        2506
Turbo Fan         1189
Unknown            463
Turbo Jet          306
Turbo Shaft         19
UNK                  1
Name: count, dtype: int64

--- Weather.Condition
Weather.Condition
VMC    51930
IMC     4746
NaN     1794
UNK      529
Unk      151
Name: count, dtype: int64

--- Number.of.Engines
Number.of.Engines
1.0    48625
2.0     7730
NaN     2313
4.0      198
3.0      174
0.0      110
Name: count, dtype: int64

--- Purpose.of.flight
Purpose.of.flight
Personal                     34930
Instructional                 8016
Unknown                       3459
Aerial Application            3453
Business                      2894
NaN                           2770
Positioning                    983
Ferry                          541
Other Work Use                 519
Aerial Observation             379
Executive/corporate            341
Public Aircraft                303
Skydiving               

In [18]:
# Engine.Type
df["Engine.Type"] = df["Engine.Type"].replace({"UNK": np.nan, "Unknown": np.nan, "LR": np.nan, "NONE": np.nan, "Geared Turbofan": "Turbo Fan"})
rare = df["Engine.Type"].value_counts()
rare = rare[rare < 50].index
df["Engine.Type"] = df["Engine.Type"].replace({r: "Other" for r in rare})

# Weather.Condition
df["Weather.Condition"] = df["Weather.Condition"].str.upper().replace({"UNK": np.nan, "UNKNOWN": np.nan})

# Number.of.Engines : 0 is not valid for a powered airplane -> NaN, then impute from make/model mode
df["Number.of.Engines"] = df["Number.of.Engines"].replace(0, np.nan)
eng_mode = df.dropna(subset=["Number.of.Engines"]).groupby("Plane.Type")["Number.of.Engines"].agg(lambda s: s.mode().iloc[0])
df["Number.of.Engines"] = df["Number.of.Engines"].fillna(df["Plane.Type"].map(eng_mode))

# Purpose.of.flight
df["Purpose.of.flight"] = (df["Purpose.of.flight"]
    .replace({"Unknown": np.nan, "Air Race show": "Air Race/show", "ASHO": "Air Race/show",
              "Executive/corporate": "Business"})
    .str.replace(r"^Public Aircraft.*", "Public Aircraft", regex=True))
pc = df["Purpose.of.flight"].value_counts()
df["Purpose.of.flight"] = df["Purpose.of.flight"].replace({p: "Other" for p in pc[pc < 50].index})

# Broad.phase.of.flight
df["Broad.phase.of.flight"] = df["Broad.phase.of.flight"].replace({"Unknown": np.nan, "Other": np.nan})

for c in ["Engine.Type", "Weather.Condition", "Number.of.Engines", "Purpose.of.flight", "Broad.phase.of.flight"]:
    print(f"--- {c} (after cleaning)")
    print(df[c].value_counts(dropna=False))
    print()

--- Engine.Type (after cleaning)
Engine.Type
Reciprocating    51603
NaN               3527
Turbo Prop        2506
Turbo Fan         1189
Turbo Jet          306
Other               19
Name: count, dtype: int64

--- Weather.Condition (after cleaning)
Weather.Condition
VMC    51930
IMC     4746
NaN     2474
Name: count, dtype: int64

--- Number.of.Engines (after cleaning)
Number.of.Engines
1.0    50018
2.0     8500
4.0      227
3.0      204
NaN      201
Name: count, dtype: int64

--- Purpose.of.flight (after cleaning)
Purpose.of.flight
Personal              34930
Instructional          8016
NaN                    6229
Aerial Application     3453
Business               3235
Positioning             983
Ferry                   541
Other Work Use          519
Public Aircraft         382
Aerial Observation      379
Skydiving               174
Flight Test             118
Banner Tow               98
Other                    93
Name: count, dtype: int64

--- Broad.phase.of.flight (after cleaning)

### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [19]:
# Non-null counts per column; keep every column with more than 20,000 non-nulls
nn = df.notna().sum().sort_values()
print(nn)
print("\nRows:", len(df))

Schedule                   6879
Air.carrier                9719
FAR.Description           17368
Longitude                 22095
Latitude                  22100
Airport.Code              35789
Airport.Name              37467
Broad.phase.of.flight     42703
Publication.Date          48956
Purpose.of.flight         52921
Engine.Type               55623
Report.Status             56008
Weather.Condition         56676
Aircraft.damage           58214
Destroyed                 58214
Registration.Number       58535
Number.of.Engines         58949
Country                   59064
Location                  59124
Aircraft.Category         59150
Event.Date                59150
Accident.Number           59150
Investigation.Type        59150
Injury.Severity           59150
Model                     59150
Make                      59150
Total.Fatal.Injuries      59150
Amateur.Built             59150
Total.Uninjured           59150
Total.Minor.Injuries      59150
Total.Serious.Injuries    59150
Event.Id

In [20]:
drop_cols = nn[nn <= 20000].index.tolist()
print("Dropping:", drop_cols)
df = df.drop(columns=drop_cols)
print("Remaining columns:", list(df.columns))
print("Shape:", df.shape)

Dropping: ['Schedule', 'Air.carrier', 'FAR.Description']
Remaining columns: ['Event.Id', 'Investigation.Type', 'Accident.Number', 'Event.Date', 'Location', 'Country', 'Latitude', 'Longitude', 'Airport.Code', 'Airport.Name', 'Injury.Severity', 'Aircraft.damage', 'Aircraft.Category', 'Registration.Number', 'Make', 'Model', 'Amateur.Built', 'Number.of.Engines', 'Engine.Type', 'Purpose.of.flight', 'Total.Fatal.Injuries', 'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured', 'Weather.Condition', 'Broad.phase.of.flight', 'Report.Status', 'Publication.Date', 'Total.Aboard', 'Fatal.Serious', 'Fatal.Serious.Frac', 'Destroyed', 'Plane.Type']
Shape: (59150, 33)


### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [21]:
df.to_csv("data/AviationData_clean.csv", index=False)
print("Saved", df.shape, "to data/AviationData_clean.csv")

Saved (59150, 33) to data/AviationData_clean.csv
